Note: Get a free AEMET API key at --> https://opendata.aemet.es/centrodedescargas/altaUsuario

# Imports

In [1]:
import os
from pathlib import Path
import requests

import pandas as pd

from dotenv import load_dotenv

load_dotenv()

AEMET_API_KEY = os.getenv("AEMET_API_KEY")

# Parameters

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_DATA_DIR = DATA_DIR / "input"

In [3]:
stations = {
    "Madrid": "3195",
    "Barcelona": "0201D",
    "Valencia": "8416",
    "Sevilla": "5783",
    "Bilbao": "1082",
    "Zaragoza": "9434",
}

# Get Data

In [ ]:
weather_dfs = []

periods = [
    ("2025-01-01T00:00:00UTC", "2025-06-30T23:59:59UTC"),
    ("2025-07-01T00:00:00UTC", "2025-12-31T23:59:59UTC"),
    ("2026-01-01T00:00:00UTC", "2026-06-30T23:59:59UTC"),
]

for start_date, end_date in periods:

    for city, station_id in stations.items():

        endpoint_aemet = (
            "https://opendata.aemet.es/opendata/api/"
            f"valores/climatologicos/diarios/datos/"
            f"fechaini/{start_date}/"
            f"fechafin/{end_date}/"
            f"estacion/{station_id}"
        )

        response_aemet = requests.get(
            endpoint_aemet,
            params={"api_key": AEMET_API_KEY},
            timeout=30,
        )

        response_aemet.raise_for_status()

        metadata_aemet = response_aemet.json()

        data_url = metadata_aemet["datos"]

        weather_response = requests.get(
            data_url,
            timeout=30,
        )

        weather_response.raise_for_status()

        weather_data = weather_response.json()

        df_city = pd.DataFrame(weather_data)

        df_city["city"] = city

        weather_dfs.append(df_city)

In [7]:
df_weather_all = (
    pd.concat(weather_dfs, ignore_index=True)
    .drop_duplicates()
)

# Preprocess

In [9]:
df_weather_all = df_weather_all[
    [
        "fecha",
        "tmed",
        "tmin",
        "tmax",
        "prec",
        "velmedia",
    ]
].copy()

In [10]:
numeric_columns = [
    "tmed",
    "tmin",
    "tmax",
    "prec",
    "velmedia",
]

for col in numeric_columns:
    df_weather_all[col] = (
        df_weather_all[col]
        .str.replace(",", ".", regex=False)
        .replace("Ip", "0")
        .astype(float)
    )

In [11]:
df_weather_all_1 = df_weather_all.groupby('fecha').agg({'tmed': 'mean', 
                                     'tmin': 'mean', 
                                     'tmax': 'mean', 
                                     'prec': 'mean', 
                                     'velmedia': 'mean'}).reset_index()

# Save Data

In [13]:
weather_output_path = (
    INPUT_DATA_DIR / "02_meteo.csv"
)

df_weather_all_1.to_csv(
    weather_output_path,
    index=False,
)